In [ ]:
## login wandb
import wandb
wandb.login()
## set up project name
import os
os.environ["WANDB_PROJECT"] = "chess-llm" 
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

In [ ]:
import unsloth
import vllm
import torch
import trl

print(vllm.__version__)
print(unsloth.__version__)
print(torch.__version__)
print(trl.__version__)

## Model

In [ ]:
from unsloth import FastLanguageModel

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Norrawee/Qwen3-4B-Thinking-2507-exp02", 
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.9, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

## Data

In [ ]:
SYSTEM_PROMPT = """You are an expert chess player.
Given the position, choose the best move in uci format enclosed by <uci_move> </uci_move>.

Position: {pad_fen}
Moves: {legal_moves_uci_list}
Your turn: {side_to_move}

Think 100 words maximum"""

In [ ]:
def pad_fen(fen):  
    parts = fen.split(" ")  
    board = parts[0]  
  
    padded_rows = []  
    for row in board.split("/"):  
        expanded = ""  
        for char in row:  
            if char.isdigit():  
                expanded += "•" * int(char)  # pad empty squares  
            else:  
                expanded += char  
        padded_rows.append(expanded)  
  
    parts[0] = "/".join(padded_rows)  
    return " ".join(parts)  
    
def format_prompt(row):  
    prompt = SYSTEM_PROMPT.format(
        side_to_move=row["side_to_move"],
        legal_moves_uci_list=" ".join(row["legal_moves_uci_list"]),
        pad_fen=pad_fen(row["fen"]),
    )    
    return [  
        {"role": "user", "content": prompt},  
    ]

In [ ]:
import pandas as pd

df = pd.read_parquet("../data/chess-exp03.parquet")
df = df[-1000:] ## test 

## preprocess
df["prompt"] = df.apply(format_prompt, axis=1)

In [ ]:
from sklearn.model_selection import train_test_split  
from datasets import Dataset  
  
# Unique boards  
unique_boards = df["board_utf"].unique()  
  
# Split boards, NOT rows  
train_boards, test_boards = train_test_split(  
    unique_boards,  
    test_size=0.1,  
    random_state=42,  
    shuffle=True,  
)  
  
# Filter rows  
train_df = df[df["board_utf"].isin(train_boards)].reset_index(drop=True)  
test_df  = df[df["board_utf"].isin(test_boards)].reset_index(drop=True)  
  
# Create HF datasets  
ds = {  
    "train": Dataset.from_pandas(train_df),  
    "test": Dataset.from_pandas(test_df),  
}  

In [ ]:
text = tokenizer.apply_chat_template(
    ds["test"][0]["prompt"], 
    tokenize=False, 
    add_generation_prompt=True,
)
print(len(tokenizer(text)["input_ids"]))
print(text)

## Reward

In [ ]:
import re
UCI_PATTERN = re.compile(r"<uci_move>(.*?)</uci_move>")  
def extract_uci(text):  
    match = UCI_PATTERN.search(text)  
    return match.group(1).strip() if match else None  
 
import chess  
import chess.engine  
# ---------------- CONFIG ----------------  
ENGINE_PATH = "stockfish"   # change if needed  
ENGINE_LIMIT = chess.engine.Limit(time=1.0, depth=16) 

  
def evaluate(fen_board, uci_move, verbose=False):
    try:
        engine = chess.engine.SimpleEngine.popen_uci(ENGINE_PATH)
        board = chess.Board(fen_board)  

        ## before 
        info_before = engine.analyse(board, ENGINE_LIMIT)  
        eval_before = info_before["score"].relative.score(mate_score=1000) 

        move = chess.Move.from_uci(uci_move) 
        board.push(move)  

        ## after
        info_after = engine.analyse(board, ENGINE_LIMIT)  
        eval_after = - info_after["score"].relative.score(mate_score=1000)
        
        engine.quit()
        
        if verbose:
            print(eval_before, eval_after)
        
        ## calculate reward
        delta = eval_after - eval_before
        if delta > 100:
            return 2
        elif delta > -100:
            return 1
        else:
            return 0
    except:
        print("Error")
        return -1
  


In [ ]:
## Test
fen_board = "k7/r7/8/R7/K7/8/8/8 w - - 0 1" 
uci_move = "a5a6"
x = evaluate(fen_board, uci_move, True)
print(f"White blunders: {x}")

fen_board = "k7/r7/8/R7/K7/8/8/8 b - - 0 1" 
uci_move = "a7a6"  
x = evaluate(fen_board, uci_move, True)
print(f"Black blunders: {x}")

In [ ]:
## Tag reward
def tag_reward(completions, **kwargs):  
    scores = []
    for completion in completions:
        text = completion[0]["content"]
        ## check uci_move
        uci_move = extract_uci(text)
        if uci_move is not None: 
            scores.append(1)
        else: 
            scores.append(0)
    return scores

## Legal move reward
def legal_reward(completions, legal_moves_uci_list, **kwargs):  
    scores = []
    for completion, legal_moves in zip(completions, legal_moves_uci_list):
        text = completion[0]["content"]
        ## check uci_move
        uci_move = extract_uci(text)
        if uci_move in legal_moves: 
            scores.append(1)
        else: 
            scores.append(0)
    return scores

## Move reward
def move_reward(completions, legal_moves_uci_list, fen, **kwargs):  
    scores = []
    for completion, legal_moves, fen_board in zip(completions, legal_moves_uci_list, fen):
        text = completion[0]["content"]
        ## check uci_move
        uci_move = extract_uci(text)
        if uci_move in legal_moves:
            score = evaluate(fen_board, uci_move)
        else:
            score = -1
        scores.append(score)
    return scores

reward_funcs = [tag_reward, legal_reward, move_reward]

## GRPO

In [ ]:
max_prompt_length = 300
max_completion_length = max_seq_length - max_prompt_length

from vllm import SamplingParams
vllm_sampling_params = SamplingParams(
    min_p = 0.1,
    top_k = -1,
    seed = 3407,
    stop = [tokenizer.eos_token],
    include_stop_str_in_output = True,
)

from trl import GRPOConfig, GRPOTrainer
training_args = GRPOConfig(
    ## algorithm
    loss_type="dapo",
    beta=0.0,
    
    ## others
    vllm_sampling_params = vllm_sampling_params,
    temperature = 0.7,
    warmup_ratio = 0.1,
    lr_scheduler_type = "linear",
    optim = "adamw_8bit",
    max_prompt_length = max_prompt_length,
    max_completion_length = max_completion_length,
    report_to = "wandb", # Can use Weights & Biases
    
    ## training params
    learning_rate=5e-5,
    num_generations=4,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    max_steps = 100,
    fp16=False,
    bf16=True,
    weight_decay = 0.001,
    
    # logging
    # eval_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    # eval_steps=5,
    save_total_limit=1,
)

In [ ]:
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs=reward_funcs,
    args = training_args,
    train_dataset = ds["train"],
    # eval_dataset = ds["test"],
)
trainer.train()

## Test

In [ ]:
text = tokenizer.apply_chat_template(
    ds["test"][3]["prompt"][:1],
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

eos_token = "<|im_end|>"
from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    temperature = 0.7,
    max_new_tokens = 1024,
    streamer = TextStreamer(tokenizer, skip_prompt = False),
    eos_token_id=tokenizer.convert_tokens_to_ids(eos_token),
)

In [ ]:
# messages = ds[0]["prompt"]

# text = tokenizer.apply_chat_template(
#     messages,
#     add_generation_prompt = True, # Must add for generation
#     tokenize = False,
# )
# from vllm import SamplingParams
# sampling_params = SamplingParams(
#     temperature = 1.0,
#     top_k = 50,
#     max_tokens = 600,
# )
# output = model.fast_generate(
#     text,
#     sampling_params = sampling_params,
#     lora_request = model.load_lora("grpo_saved_lora"),
# )[0].outputs[0].text

# print(output)

In [ ]:
# print(ds[0]["score_dict"])

## Save

In [ ]:
model.push_to_hub_merged(
    "Norrawee/Qwen3-4B-Thinking-2507-GRPO-exp03", 
    tokenizer,
    save_method = "merged_16bit", 
)